In [21]:
!pip install datasets transformers peft trl bitsandbytes accelerate -q
!pip install rouge-score nltk pandas -q
!pip install tqdm -q

In [ ]:
import os
import json
import random
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ---------------------------------------------------------------
# 0. 기본 설정
# ---------------------------------------------------------------
hf_cache = "../data/hf_cache"
os.environ["HF_HOME"]                = hf_cache
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

SEED            = 42
SAMPLE_SIZE     = None   # 테스트용 샘플 수 (전체 사용 시 None)
BASE_MODEL      = "yanolja/EEVE-Korean-Instruct-10.8B-v1.0"
DATA_PATH       = "../data/fine-tuning_data/result_clean.jsonl"   # 정제된 파일
OUTPUT_DIR      = "../final_adapter"
FINAL_DIR       = f"{OUTPUT_DIR}/final_adapter"

TEST_RATIO      = 0.1
MAX_SEQ_LENGTH  = 256   # 건설 은어 번역은 짧은 문장 → 256으로 충분
NUM_EPOCHS      = 3
LEARNING_RATE   = 2e-4
TRAIN_BATCH     = 4
EVAL_BATCH      = 8
GRAD_ACCUM      = 4     # effective batch = 16

SYSTEM_PROMPT = (
    "당신은 건설 현장의 베테랑 소장입니다.\n"
    "사용자가 입력하는 현장 은어(일본어 잔재, 오타 등)가 포함된 문장을 파악하고,\n"
    "문서 검색(RAG)에 용이하도록 깔끔한 표준어와 구체적인 설명으로 번역해주세요.\n"
    "반드시 한국어로만 답하고, 번역 결과만 출력하세요."
)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
bf16   = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"device: {device} | bf16: {bf16}")


device: cuda | bf16: True


In [ ]:

# ---------------------------------------------------------------
# 1. 데이터 정제 및 로드 (result.jsonl → result_clean.jsonl 자동 생성)
# ---------------------------------------------------------------
RAW_PATH = "../data/fine-tuning_data/result.jsonl"

# result_clean.jsonl이 없으면 자동으로 정제
if not os.path.exists(DATA_PATH):
    print("result_clean.jsonl 없음 → 자동 정제 시작...")
    src = RAW_PATH if os.path.exists(RAW_PATH) else "./result.jsonl"
    if not os.path.exists(src):
        raise FileNotFoundError(f"데이터 파일을 찾을 수 없습니다: {src}")

    clean_lines, removed = [], 0
    with open(src, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                # input/output 키가 있고 비어있지 않은 것만 유지
                if obj.get("input", "").strip() and obj.get("output", "").strip():
                    clean_lines.append(line)
                else:
                    removed += 1
            except json.JSONDecodeError:
                removed += 1

    with open(DATA_PATH, "w", encoding="utf-8") as f:
        f.write("\n".join(clean_lines))
    print(f"정제 완료: {len(clean_lines)}개 저장, {removed}개 제거")
else:
    print(f"기존 정제 파일 사용: {DATA_PATH}")

기존 정제 파일 사용: /workspace/result_clean.jsonl


In [10]:
# ---------------------------------------------------------------
# 2. JSONL 로드 및 샘플링
# ---------------------------------------------------------------
print("데이터 로드 중...")
all_data = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            all_data.append(json.loads(line))

print(f"전체 데이터: {len(all_data)}개")

# 5000개 랜덤 샘플링
random.seed(SEED)
if SAMPLE_SIZE and len(all_data) > SAMPLE_SIZE:
    sampled = random.sample(all_data, SAMPLE_SIZE)
    print(f"샘플링: {SAMPLE_SIZE}개 사용")
else:
    sampled = all_data
    print("전체 데이터 사용")

데이터 로드 중...
전체 데이터: 25616개
전체 데이터 사용


In [ ]:

# ---------------------------------------------------------------
# 3. 토크나이저 로드 및 프롬프트 구성
# ---------------------------------------------------------------
print("토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, use_fast=True, cache_dir=hf_cache
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# EEVE는 Llama 계열 → chat_template 있으면 사용, 없으면 수동 포맷
def build_prompt(user_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_text.strip()},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    return (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}"
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{user_text.strip()}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )

EOT = (
    "<|eot_id|>"
    if tokenizer.convert_tokens_to_ids("<|eot_id|>") != tokenizer.unk_token_id
    else tokenizer.eos_token
)

def build_feature(example):
    prompt_text = build_prompt(example["input"])
    full_text   = prompt_text + example["output"].strip() + EOT
    prompt_len  = len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"])
    return {
        "full_text":  full_text,
        "prompt_len": prompt_len,
        "input":      example["input"].strip(),
        "output":     example["output"].strip(),
    }


토크나이저 로드 중...


In [12]:

# ---------------------------------------------------------------
# 4. 데이터셋 분할 및 전처리
# ---------------------------------------------------------------
random.shuffle(sampled)
test_size  = max(1, int(len(sampled) * TEST_RATIO))
train_raw  = sampled[test_size:]
eval_raw   = sampled[:test_size]

print("피처 생성 중...")
train_dataset = Dataset.from_list([build_feature(d) for d in train_raw])
eval_dataset  = Dataset.from_list([build_feature(d) for d in eval_raw])
print(f"학습: {len(train_dataset)}개 | 평가: {len(eval_dataset)}개")

# 샘플 확인
print("\n[샘플 확인]")
print("input  :", train_dataset[0]["input"])
print("output :", train_dataset[0]["output"])
print("prompt_len:", train_dataset[0]["prompt_len"])

# 토큰 길이 통계
full_lens = [
    len(tokenizer(x["full_text"], add_special_tokens=False)["input_ids"])
    for x in train_dataset
]
print(f"\n토큰 길이 - min: {min(full_lens)} / mean: {np.mean(full_lens):.1f} / max: {max(full_lens)}")
if max(full_lens) > MAX_SEQ_LENGTH:
    over = sum(1 for l in full_lens if l > MAX_SEQ_LENGTH)
    print(f"⚠️  {over}개 샘플이 MAX_SEQ_LENGTH({MAX_SEQ_LENGTH}) 초과 → 잘림 처리됨")


피처 생성 중...
학습: 23055개 | 평가: 2561개

[샘플 확인]
input  : 노부찌랑 후로링 같이 쓰면 천장 마감 잘 돼?
output : 반자틀대와 플로어링을 함께 사용하면 천장 마감이 잘 이루어집니까?
prompt_len: 110

토큰 길이 - min: 106 / mean: 123.0 / max: 166


In [13]:


# ---------------------------------------------------------------
# 5. Completion-only Collator (assistant 답변만 Loss 계산)
# ---------------------------------------------------------------
class CompletionOnlyCollator:
    def __init__(self, tokenizer, max_length=256):
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __call__(self, features):
        texts       = [f["full_text"]  for f in features]
        prompt_lens = [int(f["prompt_len"]) for f in features]

        batch = self.tokenizer(
            texts,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )

        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100   # 패딩 마스킹
        for i, plen in enumerate(prompt_lens):
            labels[i, :min(plen, labels.size(1))] = -100  # 프롬프트 마스킹

        batch["labels"] = labels
        return batch

data_collator = CompletionOnlyCollator(tokenizer, max_length=MAX_SEQ_LENGTH)

# Sanity check
sanity   = data_collator([train_dataset[0]])
visible  = (sanity["labels"][0] != -100).sum().item()
print(f"\n✅ Collator 검증 - Loss 계산 토큰 수: {visible}")
if visible == 0:
    raise ValueError("❌ labels 전부 -100! 프롬프트 길이 문제를 확인하세요.")

label_text = tokenizer.decode(
    [t for t in sanity["labels"][0].tolist() if t != -100],
    skip_special_tokens=False
)
print("Loss 대상 텍스트 미리보기:", label_text[:100])


✅ Collator 검증 - Loss 계산 토큰 수: 20
Loss 대상 텍스트 미리보기: 반자틀대와 플로어링을 함께 사용하면 천장 마감이 잘 이루어집니까?<|im_end|>


In [ ]:



# ---------------------------------------------------------------
# 6. 모델 로드 (4-bit QLoRA)
# ---------------------------------------------------------------
print("\n모델 로드 중 (4-bit QLoRA)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if bf16 else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=hf_cache,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



모델 로드 중 (4-bit QLoRA)...


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

trainable params: 62,914,560 || all params: 10,867,838,976 || trainable%: 0.5789


In [16]:

# ---------------------------------------------------------------
# 7. TrainingArguments 및 Trainer
# ---------------------------------------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    bf16=bf16,
    fp16=not bf16,

    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    max_grad_norm=0.3,
    weight_decay=0.0,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


In [17]:


# ---------------------------------------------------------------
# 8. 학습
# ---------------------------------------------------------------
print("\n🚀 파인튜닝 시작!")
train_result = trainer.train()
trainer.save_state()

os.makedirs(FINAL_DIR, exist_ok=True)
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print(f"\n✅ 학습 완료!")
print(f"   best checkpoint : {trainer.state.best_model_checkpoint}")
print(f"   저장 위치       : {FINAL_DIR}")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.



🚀 파인튜닝 시작!


Epoch,Training Loss,Validation Loss
1,0.385241,0.413454
2,0.272396,0.334885
3,0.141093,0.349005


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]


✅ 학습 완료!
   best checkpoint : /workspace/EEVE-Construction-Rewriter/checkpoint-2882
   저장 위치       : /workspace/EEVE-Construction-Rewriter/final_adapter


In [24]:
from tqdm import tqdm
# ---------------------------------------------------------------
# 9. 빠른 추론 테스트
# ---------------------------------------------------------------
print("\n--- 추론 테스트 ---")
model.eval()
test_inputs = [
    "후꾸루 제대로 깔았어?",
    "네지 빠진 데 없지?",
    "가꾸목 치수 맞춰서 잘라줘",
]

for text in tqdm(test_inputs):
    prompt   = build_prompt(text)
    inputs   = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    print(f"입력: {text}")
    print(f"출력: {answer}\n")


--- 추론 테스트 ---


 33%|███▎      | 1/3 [00:05<00:10,  5.45s/it]

입력: 후꾸루 제대로 깔았어?
출력: 한지 바닥에서 완전히 접착하지 않고 띄워 바른 밑 바탕 종이를 제대로 깔았습니까?



 67%|██████▋   | 2/3 [00:07<00:03,  3.20s/it]

입력: 네지 빠진 데 없지?
출력: 나사가 빠진 곳은 없습니까?



100%|██████████| 3/3 [00:09<00:00,  3.28s/it]

입력: 가꾸목 치수 맞춰서 잘라줘
출력: 각목 치수에 맞추어 자르기를 부탁드립니다.



In [25]:
# ---------------------------------------------------------------
# 9. 추론 테스트 (10문장)
# ---------------------------------------------------------------
print("\n--- 추론 테스트 ---")
model.eval()

test_inputs = [
    "후꾸루 제대로 깔았어?",
    "네지 빠진 데 없지?",
    "가꾸목 치수 맞춰서 잘라줘",
    "단도리 잘 해놔야 해",
    "기리 맞춰서 시마이 해",
    "노가다판에서 시보리 작업 언제 해?",
    "데나오시 해야 할 부분 체크해줘",
    "합판 나라시 잘 해놨어?",
    "삿시 달기 전에 먹줄 쳐놔",
    "스라브 타설 전에 동바리 확인해",
]

# 정답이 있는 경우 함께 출력 (eval 데이터에서 같은 input 찾기)
eval_map = {d["input"]: d["output"] for d in eval_raw}

print(f"{'='*60}")
for i, text in tqdm(enumerate(test_inputs, 1)):
    prompt = build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    print(f"[{i}] 입력: {text}")
    print(f"    출력: {answer}")
    if text in eval_map:
        print(f"    정답: {eval_map[text]}")
    print()
print(f"{'='*60}")


--- 추론 테스트 ---


1it [00:04,  4.88s/it]

[1] 입력: 후꾸루 제대로 깔았어?
    출력: 한지 바닥에서 완전히 접착하지 않고 띄워 바른 밑 바탕 종이를 제대로 깔았습니까?



2it [00:06,  2.89s/it]

[2] 입력: 네지 빠진 데 없지?
    출력: 나사가 빠진 곳은 없습니까?



...

[8] 입력: 합판 나라시 잘 해놨어?
    출력: 합판을 고르기 작업을 잘 하셨나요?



9it [00:22,  2.53s/it]

[9] 입력: 삿시 달기 전에 먹줄 쳐놔
    출력: 삽을 설치하기 전에 먹줄을 쳐 놓으십시오.



10it [00:26,  2.64s/it]

[10] 입력: 스라브 타설 전에 동바리 확인해
    출력: 슬레이트 설치 전에 이음매 보강을 확인하십시오.



In [27]:
train_df_columns = [
    "sample_id",
    "split",      # train / eval
    "input",
    "output"
]
eval_df_columns = [
    "sample_id",
    "input",
    "reference",      # 정답
    "prediction",     # 모델 생성문
    "rouge1_f",
    "rouge2_f",
    "rougeL_f",
    "bleu4",
    "ref_len",
    "pred_len",
    "exact_match"
]

In [ ]:


import re
import pandas as pd
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 한국어라서 stemmer는 끄는 쪽이 무난
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False
)
smooth = SmoothingFunction().method1

def normalize_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def bleu_tokenize(text: str):
    # 일단 공백 기준
    # 나중에 Mecab/Okt 쓰려면 여기만 바꾸면 됨
    return normalize_text(text).split()

rows = []

model.eval()

for i, ex in tqdm(enumerate(eval_dataset)):
    input_text = ex["input"]
    ref_raw = ex["output"]

    prompt = build_prompt(input_text)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(device)

    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    pred_raw = tokenizer.decode(
        gen[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    reference = normalize_text(ref_raw)
    prediction = normalize_text(pred_raw)

    rouge_score = rouge.score(reference, prediction)

    ref_tokens = bleu_tokenize(reference)
    pred_tokens = bleu_tokenize(prediction)

    if len(pred_tokens) == 0:
        bleu4 = 0.0
    else:
        bleu4 = sentence_bleu(
            [ref_tokens],          # list(list(str))
            pred_tokens,           # list(str)
            smoothing_function=smooth
        )

    rows.append({
        "sample_id": i,
        "input": input_text,
        "reference": reference,
        "prediction": prediction,
        "rouge1_f": rouge_score["rouge1"].fmeasure,
        "rouge2_f": rouge_score["rouge2"].fmeasure,
        "rougeL_f": rouge_score["rougeL"].fmeasure,
        "bleu4": bleu4,
        "ref_len": len(ref_tokens),
        "pred_len": len(pred_tokens),
        "exact_match": int(reference == prediction),
    })

eval_df = pd.DataFrame(rows)

summary_df = pd.DataFrame([{
    "rouge1_f": eval_df["rouge1_f"].mean(),
    "rouge2_f": eval_df["rouge2_f"].mean(),
    "rougeL_f": eval_df["rougeL_f"].mean(),
    "bleu4": eval_df["bleu4"].mean(),
    "exact_match_rate": eval_df["exact_match"].mean(),
}]).round(4)

display(eval_df.head())
display(summary_df)

52it [02:42,  3.41s/it]